# Gold-location stratified communication analysis

This offline analysis uses the **actual frozen BM25 top-six retrieval and P sharding**, not a hypothetical ten-document partition. Gold is used only to classify evidence ownership after inference. Keep Disjoint-Gold, Co-located-Gold, Retrieval-Missing-Gold and Undefined-Gold separate; retain the pooled primary analysis.

Report question counts and type composition, F1/EM and measured costs, paired effects within each group, and Disjoint-Gold minus Co-located-Gold effect heterogeneity. E5 averages optimizer seeds within questions. Estimates remain exploratory; no positive or massive effect is presumed. R-condition results use the same P-defined question strata to preserve the question-paired interaction.

For 3/3/4 shards the exact uniform co-location probability is 12/45=26.67%. For the actual six-document 2/2/2 shards it is 3/15=20%, conditional on both gold articles being retrieved. Missing retrieval is a distinct failure mode. See `analysis/STRATIFICATION_PLAN.md`.

Active sample-size revision: **100 fixed training questions, 30 fixed validation questions, and 300 unchanged separate held-out questions**. Three collection epochs and three optimizer seeds remain. Previous 300/100 artifacts are preserved as a separate study; their predictions and scores did not select the subsets. Reused provider responses retain their original provenance and costs. This notebook has not yet executed the new study to completion.

In [ ]:
from pathlib import Path
import sys,json
ROOT=Path.cwd() if (Path.cwd()/'protocol.py').exists() else Path.cwd()/'WEBExtension'
ROOT=ROOT.resolve();sys.path.insert(0,str(ROOT))
from analysis.gold_strata import analyze
from experiments import load_config,freeze
from controller_study import study_directory
CFG=load_config();SETTINGS=json.loads((ROOT/'controller_config.json').read_text())
RUN_DIR,_=freeze(CFG)
STUDY_DIR,_=study_directory(CFG,SETTINGS)
print('Inference run:',RUN_DIR)
print('Controller study:',STUDY_DIR)


In [ ]:
from IPython.display import display
for experiment,directory in [(e,RUN_DIR) for e in ['E1','E2','E3','E4']]+[('E5',STUDY_DIR)]:
    status=directory/(experiment+'_status.json')
    if status.exists() and json.loads(status.read_text()).get('status')=='COMPLETE':
        summary,statistics=analyze(directory,experiment)
        print(experiment,statistics['counts'])
        display(summary)
        print(json.dumps(statistics['effects'],indent=2))
        print(json.dumps(statistics['heterogeneity'],indent=2))
    else:
        print(experiment,'NO COMPLETE RESULTS: no stratum effect is estimated')
